<a href="https://colab.research.google.com/github/Sofiacontecha/Taller-graficas-/blob/main/FoodWise.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🥗 FoodWise
### Cocina con lo que ya tienes en casa

**Integrantes:** Sofia Contecha, Sarah Bonilla, Emiliana Sepulveda y Samuel Roa
**Opción elegida:** A – Una aplicación sencilla en Python

**Pregunta orientadora:** ¿Cómo puede una aplicación ayudar a los usuarios a decidir qué cocinar con los ingredientes que ya tienen, para reducir el desperdicio de alimentos y el gasto en compras?

FoodWise permite:
1. Registrar los ingredientes que hay en casa, con cantidad y fecha de vencimiento.
2. Ver qué alimentos hay que consumir pronto.
3. Recibir recetas sugeridas que aprovechan lo que ya se tiene, priorizando lo que está por vencer.
4. Ver el costo aproximado de cada receta y los ingredientes que faltan.
5. Generar una lista de compras solo con lo que hace falta.

**Cómo usarlo:** ejecuta las celdas en orden (*Entorno de ejecución → Ejecutar todas* en Colab). La sección 6 muestra una demostración automática y la sección 7 abre el menú interactivo.

## 1. Preparar los datos

### 1.1 Tabla de precios
Cada ingrediente tiene una **unidad de medida** y un **precio aproximado por unidad** en pesos colombianos (COP), tomado como referencia de supermercados y plazas de mercado. Por ejemplo, el arroz se mide en gramos y cuesta unos $4.500 el kilo, es decir, $4,5 por gramo.

Se asume que la sal, la pimienta y el agua siempre están disponibles, por eso no aparecen en las recetas.

In [3]:
from datetime import date, datetime, timedelta

# nombre del ingrediente -> unidad de medida y precio aproximado por unidad (COP)
PRECIOS = {
    "huevo":          {"unidad": "und",    "precio": 650},    # cubeta de 30 ≈ $19.500
    "arroz":          {"unidad": "g",      "precio": 4.5},    # 1 kg ≈ $4.500
    "tomate":         {"unidad": "und",    "precio": 800},
    "cebolla":        {"unidad": "und",    "precio": 800},
    "ajo":            {"unidad": "diente", "precio": 300},
    "papa":           {"unidad": "g",      "precio": 3.5},    # 1 kg ≈ $3.500
    "pasta":          {"unidad": "g",      "precio": 9},      # 500 g ≈ $4.500
    "leche":          {"unidad": "ml",     "precio": 4.5},    # 1 L ≈ $4.500
    "queso":          {"unidad": "g",      "precio": 32},     # 250 g ≈ $8.000
    "pan":            {"unidad": "tajada", "precio": 400},
    "pollo":          {"unidad": "g",      "precio": 18},     # 1 kg ≈ $18.000
    "carne molida":   {"unidad": "g",      "precio": 26},     # 1 kg ≈ $26.000
    "frijol":         {"unidad": "g",      "precio": 12},     # 500 g ≈ $6.000
    "lenteja":        {"unidad": "g",      "precio": 8},      # 500 g ≈ $4.000
    "zanahoria":      {"unidad": "und",    "precio": 500},
    "platano":        {"unidad": "und",    "precio": 1500},
    "aguacate":       {"unidad": "und",    "precio": 3000},
    "atun":           {"unidad": "lata",   "precio": 6500},
    "mantequilla":    {"unidad": "g",      "precio": 40},
    "banano":         {"unidad": "und",    "precio": 400},
    "avena":          {"unidad": "g",      "precio": 10},
    "azucar":         {"unidad": "g",      "precio": 4},
    "aceite":         {"unidad": "ml",     "precio": 12},     # 1 L ≈ $12.000
    "arepa":          {"unidad": "und",    "precio": 700},
    "jamon":          {"unidad": "g",      "precio": 35},
    "lechuga":        {"unidad": "und",    "precio": 3000},
    "limon":          {"unidad": "und",    "precio": 300},
    "pimenton":       {"unidad": "und",    "precio": 1500},
    "crema de leche": {"unidad": "ml",     "precio": 16},
    "salchicha":      {"unidad": "und",    "precio": 1200},
}

print("Ingredientes en la tabla de precios:", len(PRECIOS))

Ingredientes en la tabla de precios: 30


### 1.2 Base de recetas
Las recetas se guardan como una **lista de diccionarios**. Cada receta tiene su nombre, las porciones, los ingredientes con la cantidad necesaria (en la misma unidad de la tabla de precios) y una preparación corta.

In [4]:
RECETAS = [
    {"nombre": "Arroz con huevo y tomate", "porciones": 2,
     "ingredientes": {"arroz": 150, "huevo": 2, "tomate": 1, "cebolla": 0.5, "aceite": 15},
     "preparacion": "Cocina el arroz. Sofríe la cebolla y el tomate picados, agrega los huevos revueltos y mezcla con el arroz."},
    {"nombre": "Tortilla con salsa de tomate", "porciones": 2,
     "ingredientes": {"huevo": 3, "tomate": 2, "cebolla": 0.5, "aceite": 15},
     "preparacion": "Bate los huevos y cocínalos en una sartén con aceite. Aparte, cocina el tomate y la cebolla picados hasta hacer una salsa y sírvela encima."},
    {"nombre": "Huevos pericos con arepa", "porciones": 1,
     "ingredientes": {"huevo": 2, "tomate": 1, "cebolla": 0.5, "aceite": 10, "arepa": 2},
     "preparacion": "Sofríe tomate y cebolla, agrega los huevos y revuelve. Acompaña con arepa asada."},
    {"nombre": "Arroz con pollo", "porciones": 4,
     "ingredientes": {"arroz": 200, "pollo": 300, "zanahoria": 1, "cebolla": 1, "pimenton": 0.5, "ajo": 2, "aceite": 20},
     "preparacion": "Cocina y desmecha el pollo. Sofríe cebolla, ajo, pimentón y zanahoria, agrega el arroz, el pollo y el agua, y cocina hasta que seque."},
    {"nombre": "Pasta en salsa de tomate", "porciones": 2,
     "ingredientes": {"pasta": 200, "tomate": 3, "cebolla": 0.5, "ajo": 2, "aceite": 15},
     "preparacion": "Cocina la pasta. Licúa o pica el tomate y cocínalo con cebolla y ajo. Mezcla con la pasta."},
    {"nombre": "Pasta con atún", "porciones": 2,
     "ingredientes": {"pasta": 200, "atun": 1, "crema de leche": 100, "cebolla": 0.5},
     "preparacion": "Cocina la pasta. Sofríe la cebolla, agrega el atún y la crema de leche, y mezcla con la pasta."},
    {"nombre": "Sopa de lentejas", "porciones": 4,
     "ingredientes": {"lenteja": 250, "papa": 200, "zanahoria": 1, "cebolla": 0.5, "tomate": 1, "ajo": 2},
     "preparacion": "Cocina las lentejas en agua. Agrega papa y zanahoria en cubos y un hogao de tomate, cebolla y ajo. Cocina hasta que todo ablande."},
    {"nombre": "Frijoles con arroz y tajada", "porciones": 4,
     "ingredientes": {"frijol": 250, "arroz": 150, "platano": 1, "cebolla": 0.5, "tomate": 1},
     "preparacion": "Remoja y cocina los frijoles con hogao. Sirve con arroz y tajadas de plátano frito."},
    {"nombre": "Papas sudadas", "porciones": 3,
     "ingredientes": {"papa": 500, "tomate": 2, "cebolla": 1, "aceite": 15},
     "preparacion": "Haz un hogao con tomate y cebolla, agrega la papa en trozos y un poco de agua. Tapa y cocina hasta que ablande."},
    {"nombre": "Ensalada de atún", "porciones": 2,
     "ingredientes": {"atun": 1, "lechuga": 0.5, "tomate": 1, "aguacate": 0.5, "limon": 1},
     "preparacion": "Pica la lechuga, el tomate y el aguacate. Agrega el atún y aliña con limón y sal."},
    {"nombre": "Sándwich de jamón y queso", "porciones": 2,
     "ingredientes": {"pan": 4, "jamon": 60, "queso": 60, "tomate": 1, "lechuga": 0.25},
     "preparacion": "Arma los sándwiches con jamón, queso, tomate y lechuga. Puedes calentarlos en sartén."},
    {"nombre": "Arepa con huevo y queso", "porciones": 1,
     "ingredientes": {"arepa": 2, "huevo": 2, "queso": 50, "mantequilla": 10},
     "preparacion": "Asa las arepas con mantequilla, fríe los huevos y sirve todo con queso encima."},
    {"nombre": "Pancakes de banano y avena", "porciones": 2,
     "ingredientes": {"banano": 2, "avena": 100, "huevo": 2, "leche": 100},
     "preparacion": "Licúa todos los ingredientes y cocina porciones pequeñas en una sartén caliente."},
    {"nombre": "Colada de avena", "porciones": 2,
     "ingredientes": {"avena": 80, "leche": 400, "azucar": 30},
     "preparacion": "Cocina la avena en la leche a fuego bajo, revolviendo, y endulza con azúcar."},
    {"nombre": "Carne molida con papa", "porciones": 3,
     "ingredientes": {"carne molida": 300, "papa": 300, "tomate": 1, "cebolla": 1, "ajo": 2, "aceite": 15},
     "preparacion": "Sofríe cebolla, ajo y tomate, agrega la carne y luego la papa en cubos pequeños. Cocina tapado hasta que la papa ablande."},
    {"nombre": "Pollo a la plancha con ensalada", "porciones": 2,
     "ingredientes": {"pollo": 300, "lechuga": 0.5, "tomate": 1, "limon": 1, "aceite": 10},
     "preparacion": "Adoba el pollo con limón y sal y ásalo en sartén. Sirve con ensalada de lechuga y tomate."},
    {"nombre": "Patacones con hogao", "porciones": 2,
     "ingredientes": {"platano": 2, "tomate": 2, "cebolla": 1, "aceite": 100},
     "preparacion": "Fríe trozos de plátano, aplánalos y vuelve a freírlos. Sirve con hogao de tomate y cebolla."},
    {"nombre": "Salchipapa casera", "porciones": 2,
     "ingredientes": {"salchicha": 3, "papa": 400, "aceite": 150},
     "preparacion": "Corta las papas en bastones y fríelas. Fríe las salchichas en rodajas y mezcla."},
    {"nombre": "Guacamole con arepa", "porciones": 2,
     "ingredientes": {"aguacate": 1, "tomate": 1, "cebolla": 0.5, "limon": 1, "arepa": 2},
     "preparacion": "Machaca el aguacate y mézclalo con tomate y cebolla picados y limón. Sirve con arepa."},
    {"nombre": "Puré de papa", "porciones": 3,
     "ingredientes": {"papa": 500, "leche": 100, "mantequilla": 30},
     "preparacion": "Cocina las papas peladas, aplástalas y mezcla con leche caliente y mantequilla."},
]

# Verificación: todos los ingredientes de las recetas deben estar en la tabla de precios
for receta in RECETAS:
    for ingrediente in receta["ingredientes"]:
        if ingrediente not in PRECIOS:
            print("Falta el precio de:", ingrediente, "en la receta", receta["nombre"])

print("Recetas cargadas:", len(RECETAS))

Recetas cargadas: 20


### 1.3 Funciones de apoyo
Pequeñas funciones para mostrar los precios y las cantidades de forma legible.

In [5]:
def formato_pesos(valor):
    """Convierte un número en texto de pesos colombianos: 12500 -> $12.500"""
    texto = f"{round(valor):,}"
    return "$" + texto.replace(",", ".")


def texto_porciones(n):
    if n == 1:
        return "1 porción"
    return f"{n} porciones"


def formato_cantidad(cantidad, nombre):
    """Muestra la cantidad con su unidad: 150 g, 2 und, 0.5 und"""
    unidad = PRECIOS[nombre]["unidad"]
    if cantidad == int(cantidad):
        cantidad = int(cantidad)
    else:
        cantidad = round(cantidad, 2)
    return f"{cantidad} {unidad}"


print(formato_pesos(12500), "|", formato_cantidad(150, "arroz"), "|", formato_cantidad(0.5, "cebolla"))

$12.500 | 150 g | 0.5 und


## 2. Registrar el inventario

El inventario es un **diccionario**: la llave es el nombre del ingrediente y el valor guarda la cantidad y la fecha de vencimiento. Si el ingrediente ya estaba registrado, se suma la cantidad y se conserva la fecha de vencimiento más cercana.

Las fechas se escriben como `DD/MM/AAAA`. Si un alimento no se vence pronto (arroz, aceite), se puede dejar la fecha en blanco.

In [6]:
inventario = {}   # ejemplo: {"huevo": {"cantidad": 6, "vence": date(2026, 11, 25)}}


def agregar_ingrediente(nombre, cantidad, vence=None):
    """Agrega un ingrediente al inventario. Devuelve True si se pudo agregar."""
    nombre = nombre.strip().lower()

    if nombre not in PRECIOS:
        print(f"  ⚠ '{nombre}' no está en la base de FoodWise.")
        print("    Ingredientes disponibles:", ", ".join(sorted(PRECIOS)))
        return False

    if cantidad <= 0:
        print("  ⚠ La cantidad debe ser mayor que cero.")
        return False

    if nombre in inventario:
        inventario[nombre]["cantidad"] += cantidad
        fecha_actual = inventario[nombre]["vence"]
        if vence is not None and (fecha_actual is None or vence < fecha_actual):
            inventario[nombre]["vence"] = vence
    else:
        inventario[nombre] = {"cantidad": cantidad, "vence": vence}
    return True


def eliminar_ingrediente(nombre):
    nombre = nombre.strip().lower()
    if nombre in inventario:
        del inventario[nombre]
        print(f"  '{nombre}' se eliminó del inventario.")
    else:
        print(f"  '{nombre}' no está en el inventario.")


def leer_fecha(texto):
    """Convierte 'DD/MM/AAAA' en fecha. Devuelve None si el texto está vacío o no es válido."""
    texto = texto.strip()
    if texto == "":
        return None
    try:
        return datetime.strptime(texto, "%d/%m/%Y").date()
    except ValueError:
        return "error"


def registrar_ingrediente_usuario():
    """Pide al usuario los datos de un ingrediente con input() y lo agrega al inventario."""
    nombre = input("Nombre del ingrediente: ").strip().lower()
    if nombre not in PRECIOS:
        agregar_ingrediente(nombre, 1)   # muestra el mensaje con los ingredientes válidos
        return

    unidad = PRECIOS[nombre]["unidad"]
    cantidad = None
    while cantidad is None:
        texto = input(f"Cantidad (en {unidad}): ").replace(",", ".")
        try:
            cantidad = float(texto)
            if cantidad <= 0:
                print("  La cantidad debe ser mayor que cero.")
                cantidad = None
        except ValueError:
            print("  Escribe un número, por ejemplo 2 o 500.")

    vence = "error"
    while vence == "error":
        vence = leer_fecha(input("Fecha de vencimiento DD/MM/AAAA (Enter si no se vence pronto): "))
        if vence == "error":
            print("  Fecha no válida. Ejemplo: 28/11/2026")

    if agregar_ingrediente(nombre, cantidad, vence):
        print(f"  ✓ Se registró {formato_cantidad(cantidad, nombre)} de {nombre}.")

## 3. Funciones principales

### 3.1 Alimentos por consumir pronto
Se calculan los **días restantes** hasta la fecha de vencimiento y, con una condicional, se clasifica cada alimento:

| Días restantes | Estado |
|---|---|
| menos de 0 | ❌ Vencido (no se usa en las recetas) |
| 0 a 3 | 🔴 Consumir pronto (se prioriza en las recetas) |
| 4 a 7 | 🟡 Consumir esta semana |
| más de 7 | 🟢 En buen estado |
| sin fecha | ⚪ No perecedero |

In [7]:
DIAS_PRONTO = 3   # un alimento con 3 días o menos se considera "por vencer"


def dias_restantes(vence):
    if vence is None:
        return None
    return (vence - date.today()).days


def estado_alimento(dias):
    if dias is None:
        return "⚪ No perecedero"
    elif dias < 0:
        return "❌ Vencido"
    elif dias <= DIAS_PRONTO:
        return "🔴 Consumir pronto"
    elif dias <= 7:
        return "🟡 Esta semana"
    else:
        return "🟢 En buen estado"


def clave_dias(nombre):
    """Sirve para ordenar: primero lo que vence antes; lo que no vence va al final."""
    dias = dias_restantes(inventario[nombre]["vence"])
    if dias is None:
        return 9999
    return dias


def mostrar_inventario():
    if len(inventario) == 0:
        print("El inventario está vacío. Registra ingredientes con la opción 1.")
        return
    print(f"\n{'INGREDIENTE':<16}{'CANTIDAD':<14}{'VENCE':<13}{'DÍAS':<7}ESTADO")
    print("-" * 70)
    for nombre in sorted(inventario, key=clave_dias):
        datos = inventario[nombre]
        dias = dias_restantes(datos["vence"])
        if datos["vence"] is None:
            fecha_txt, dias_txt = "-", "-"
        else:
            fecha_txt, dias_txt = datos["vence"].strftime("%d/%m/%Y"), str(dias)
        print(f"{nombre:<16}{formato_cantidad(datos['cantidad'], nombre):<14}{fecha_txt:<13}{dias_txt:<7}{estado_alimento(dias)}")


def alimentos_por_vencer():
    """Devuelve dos listas: los alimentos por vencer (0 a 3 días) y los vencidos."""
    por_vencer = []
    vencidos = []
    for nombre in sorted(inventario, key=clave_dias):
        dias = dias_restantes(inventario[nombre]["vence"])
        if dias is None:
            continue
        if dias < 0:
            vencidos.append(nombre)
        elif dias <= DIAS_PRONTO:
            por_vencer.append(nombre)
    return por_vencer, vencidos


def mostrar_por_vencer():
    por_vencer, vencidos = alimentos_por_vencer()
    if len(por_vencer) == 0 and len(vencidos) == 0:
        print("✓ No tienes alimentos por vencer en los próximos días.")
        return
    if len(por_vencer) > 0:
        print("\n🔴 Consume pronto estos alimentos:")
        for nombre in por_vencer:
            dias = dias_restantes(inventario[nombre]["vence"])
            if dias == 0:
                cuando = "vence HOY"
            elif dias == 1:
                cuando = "vence mañana"
            else:
                cuando = f"vence en {dias} días"
            print(f"   - {nombre} ({formato_cantidad(inventario[nombre]['cantidad'], nombre)}): {cuando}")
    if len(vencidos) > 0:
        print("\n❌ Estos alimentos ya se vencieron (revísalos antes de usarlos):")
        for nombre in vencidos:
            print(f"   - {nombre}")

### 3.2 Recetas sugeridas, costo e ingredientes faltantes
Para cada receta, un **ciclo** recorre sus ingredientes y los compara con el inventario:

- Si hay suficiente cantidad, el ingrediente está completo ✓.
- Si no hay o no alcanza, se calcula **cuánto falta** y cuánto costaría comprarlo.
- Si el ingrediente está **por vencer**, la receta gana puntos extra.

**Puntaje de cada receta** = % de ingredientes que ya tienes + 15 puntos por cada ingrediente por vencer que aprovecha.

Así, la búsqueda parte de lo que el usuario ya tiene y no de la receta que quiere hacer. Las recetas se ordenan de mayor a menor puntaje y, si empatan, va primero la que requiere gastar menos.

In [8]:
PUNTOS_POR_VENCER = 15


def cantidad_disponible(nombre, inv):
    """Cantidad que se puede usar de un ingrediente (los vencidos no cuentan)."""
    if nombre not in inv:
        return 0
    dias = dias_restantes(inv[nombre]["vence"])
    if dias is not None and dias < 0:
        return 0
    return inv[nombre]["cantidad"]


def analizar_receta(receta, inv):
    """Compara una receta con el inventario y calcula cobertura, faltantes y costos."""
    tiene = []
    faltantes = {}
    usa_por_vencer = []
    usa_del_inventario = 0
    costo_total = 0
    costo_faltante = 0

    for nombre, necesita in receta["ingredientes"].items():
        precio = PRECIOS[nombre]["precio"]
        costo_total = costo_total + necesita * precio
        disponible = cantidad_disponible(nombre, inv)

        if disponible > 0:
            usa_del_inventario += 1
            dias = dias_restantes(inv[nombre]["vence"])
            if dias is not None and dias <= DIAS_PRONTO:
                usa_por_vencer.append(nombre)

        if disponible >= necesita:
            tiene.append(nombre)
        else:
            falta = necesita - disponible
            faltantes[nombre] = falta
            costo_faltante = costo_faltante + falta * precio

    cobertura = len(tiene) / len(receta["ingredientes"]) * 100
    puntaje = cobertura + PUNTOS_POR_VENCER * len(usa_por_vencer)

    return {
        "receta": receta,
        "tiene": tiene,
        "faltantes": faltantes,
        "usa_por_vencer": usa_por_vencer,
        "usa_del_inventario": usa_del_inventario,
        "cobertura": cobertura,
        "puntaje": puntaje,
        "costo_total": costo_total,
        "costo_faltante": costo_faltante,
    }


def clave_orden(analisis):
    # mayor puntaje primero; si empatan, menor costo de lo que falta primero
    return (-analisis["puntaje"], analisis["costo_faltante"])


def sugerir_recetas(inv, maximo=5):
    """Devuelve las mejores recetas según el inventario."""
    resultados = []
    for receta in RECETAS:
        analisis = analizar_receta(receta, inv)
        if analisis["usa_del_inventario"] > 0:      # solo recetas que usen algo de lo que hay
            resultados.append(analisis)
    resultados.sort(key=clave_orden)
    return resultados[:maximo]


def mostrar_sugerencias(sugerencias):
    if len(sugerencias) == 0:
        print("No hay recetas que usen tus ingredientes. Registra más ingredientes en el inventario.")
        return
    print("\n🍽  RECETAS SUGERIDAS CON LO QUE TIENES")
    print("=" * 60)
    for i in range(len(sugerencias)):
        a = sugerencias[i]
        print(f"\n{i + 1}. {a['receta']['nombre']}  ({texto_porciones(a['receta']['porciones'])})")
        print(f"   Tienes {len(a['tiene'])} de {len(a['receta']['ingredientes'])} ingredientes ({a['cobertura']:.0f}%)")
        if len(a["usa_por_vencer"]) > 0:
            print("   ♻ Aprovecha lo que está por vencer:", ", ".join(a["usa_por_vencer"]))
        print(f"   Costo aproximado de la receta: {formato_pesos(a['costo_total'])}")
        if len(a["faltantes"]) == 0:
            print("   ✓ ¡Tienes todo! No necesitas comprar nada.")
        else:
            textos = []
            for nombre, falta in a["faltantes"].items():
                textos.append(f"{nombre} ({formato_cantidad(falta, nombre)})")
            print("   Te falta:", ", ".join(textos))
            print(f"   Gasto para completarla: {formato_pesos(a['costo_faltante'])}")


def mostrar_detalle_receta(a):
    receta = a["receta"]
    print(f"\n📖 {receta['nombre'].upper()}  ({texto_porciones(receta['porciones'])})")
    print("-" * 60)
    for nombre, necesita in receta["ingredientes"].items():
        costo = necesita * PRECIOS[nombre]["precio"]
        if nombre in a["faltantes"]:
            marca = f"✗ falta {formato_cantidad(a['faltantes'][nombre], nombre)}"
        else:
            marca = "✓ lo tienes"
        print(f"  {nombre:<16}{formato_cantidad(necesita, nombre):<12}{formato_pesos(costo):>9}   {marca}")
    print("-" * 60)
    print(f"  Costo aproximado total: {formato_pesos(a['costo_total'])}")
    print(f"  Lo que tendrías que comprar: {formato_pesos(a['costo_faltante'])}")
    print("\nPreparación:", receta["preparacion"])

### 3.3 Lista de compras automática
El usuario elige qué recetas quiere preparar y FoodWise junta todo lo que falta en **una sola lista**. Si dos recetas usan el mismo ingrediente, el programa va "gastando" una copia del inventario receta por receta, para no contar dos veces lo que ya se tiene.

In [9]:
def copiar_inventario(inv):
    copia = {}
    for nombre in inv:
        copia[nombre] = {"cantidad": inv[nombre]["cantidad"], "vence": inv[nombre]["vence"]}
    return copia


def generar_lista_compras(recetas_elegidas, inv):
    """Devuelve un diccionario {ingrediente: cantidad que hay que comprar}."""
    inv_simulado = copiar_inventario(inv)
    lista = {}
    for receta in recetas_elegidas:
        for nombre, necesita in receta["ingredientes"].items():
            disponible = cantidad_disponible(nombre, inv_simulado)
            usado = min(disponible, necesita)
            if usado > 0:
                inv_simulado[nombre]["cantidad"] -= usado
            falta = necesita - usado
            if falta > 0:
                lista[nombre] = lista.get(nombre, 0) + falta
    return lista


def mostrar_lista_compras(lista, recetas_elegidas):
    print("\n🛒 LISTA DE COMPRAS")
    print("Para preparar:", ", ".join([r["nombre"] for r in recetas_elegidas]))
    print("-" * 50)
    if len(lista) == 0:
        print("  ¡No necesitas comprar nada! Ya tienes todo.")
        return
    total = 0
    for nombre in sorted(lista):
        costo = lista[nombre] * PRECIOS[nombre]["precio"]
        total += costo
        print(f"  [ ] {nombre:<16}{formato_cantidad(lista[nombre], nombre):<12}{formato_pesos(costo):>10}")
    print("-" * 50)
    print(f"  Total aproximado: {formato_pesos(total):>23}")
    print("\n  * Los precios son aproximados. En la tienda algunos productos")
    print("    se venden por paquete (ej. 1 kg de arroz), así que el gasto real puede variar.")

## 4. Inventario de ejemplo
Para la demostración, se carga un inventario con fechas calculadas a partir del día de hoy, de modo que siempre haya alimentos por vencer.

In [10]:
def cargar_inventario_ejemplo():
    inventario.clear()
    hoy = date.today()
    agregar_ingrediente("huevo", 6, hoy + timedelta(days=2))
    agregar_ingrediente("arroz", 1000)
    agregar_ingrediente("tomate", 3, hoy + timedelta(days=1))
    agregar_ingrediente("cebolla", 2, hoy + timedelta(days=10))
    agregar_ingrediente("papa", 800, hoy + timedelta(days=6))
    agregar_ingrediente("leche", 500, hoy + timedelta(days=3))
    agregar_ingrediente("queso", 200, hoy + timedelta(days=5))
    agregar_ingrediente("arepa", 4, hoy + timedelta(days=4))
    agregar_ingrediente("aceite", 500)
    agregar_ingrediente("banano", 2, hoy - timedelta(days=1))   # ya vencido
    print("✓ Inventario de ejemplo cargado.")

## 5. Pruebas

### Caso de la propuesta: huevos, arroz y tomate
Solo con estos tres ingredientes, FoodWise debería sugerir *Arroz con huevo y tomate* y *Tortilla con salsa de tomate*.

In [11]:
inventario.clear()
hoy = date.today()
agregar_ingrediente("huevo", 6, hoy + timedelta(days=2))
agregar_ingrediente("arroz", 500)
agregar_ingrediente("tomate", 3, hoy + timedelta(days=1))

sugerencias = sugerir_recetas(inventario, maximo=3)
mostrar_sugerencias(sugerencias)

nombres = [a["receta"]["nombre"] for a in sugerencias]
print("\nPrueba superada ✓" if "Arroz con huevo y tomate" in nombres and "Tortilla con salsa de tomate" in nombres
      else "\nRevisar la prueba ✗")


🍽  RECETAS SUGERIDAS CON LO QUE TIENES

1. Arroz con huevo y tomate  (2 porciones)
   Tienes 3 de 5 ingredientes (60%)
   ♻ Aprovecha lo que está por vencer: huevo, tomate
   Costo aproximado de la receta: $3.355
   Te falta: cebolla (0.5 und), aceite (15 ml)
   Gasto para completarla: $580

2. Tortilla con salsa de tomate  (2 porciones)
   Tienes 2 de 4 ingredientes (50%)
   ♻ Aprovecha lo que está por vencer: huevo, tomate
   Costo aproximado de la receta: $4.130
   Te falta: cebolla (0.5 und), aceite (15 ml)
   Gasto para completarla: $580

3. Huevos pericos con arepa  (1 porción)
   Tienes 2 de 5 ingredientes (40%)
   ♻ Aprovecha lo que está por vencer: huevo, tomate
   Costo aproximado de la receta: $4.020
   Te falta: cebolla (0.5 und), aceite (10 ml), arepa (2 und)
   Gasto para completarla: $1.920

Prueba superada ✓


### Otras pruebas rápidas

In [12]:
# 1) Los alimentos vencidos no se cuentan como disponibles
inventario.clear()
agregar_ingrediente("leche", 500, date.today() - timedelta(days=2))
print("Leche vencida disponible:", cantidad_disponible("leche", inventario), "(debe ser 0)")

# 2) Un ingrediente que no existe en la base no se agrega
inventario.clear()
agregar_ingrediente("caviar", 1)
print("Inventario después de intentar agregar caviar:", inventario, "(debe estar vacío)")

# 3) Si se registra el mismo ingrediente dos veces, se suman las cantidades
agregar_ingrediente("huevo", 6, date.today() + timedelta(days=5))
agregar_ingrediente("huevo", 4, date.today() + timedelta(days=2))
print("Huevos:", inventario["huevo"]["cantidad"], "(debe ser 10), vence en",
      dias_restantes(inventario["huevo"]["vence"]), "días (debe ser 2, la fecha más cercana)")

# 4) La lista de compras no cuenta dos veces lo que ya se tiene
inventario.clear()
agregar_ingrediente("huevo", 3)
lista = generar_lista_compras([RECETAS[0], RECETAS[1]], inventario)   # usan 2 + 3 huevos
print("Huevos por comprar:", lista.get("huevo", 0), "(debe ser 2)")

Leche vencida disponible: 0 (debe ser 0)
  ⚠ 'caviar' no está en la base de FoodWise.
    Ingredientes disponibles: aceite, aguacate, ajo, arepa, arroz, atun, avena, azucar, banano, carne molida, cebolla, crema de leche, frijol, huevo, jamon, leche, lechuga, lenteja, limon, mantequilla, pan, papa, pasta, pimenton, platano, pollo, queso, salchicha, tomate, zanahoria
Inventario después de intentar agregar caviar: {} (debe estar vacío)
Huevos: 10 (debe ser 10), vence en 2 días (debe ser 2, la fecha más cercana)
Huevos por comprar: 2 (debe ser 2)


## 6. Demostración completa
Esta celda muestra todo el recorrido sin necesidad de escribir nada: inventario, alimentos por vencer, recetas sugeridas, detalle de la mejor receta y lista de compras para las tres primeras.

In [13]:
cargar_inventario_ejemplo()

print("\n📦 INVENTARIO")
mostrar_inventario()

mostrar_por_vencer()

sugerencias = sugerir_recetas(inventario)
mostrar_sugerencias(sugerencias)

mostrar_detalle_receta(sugerencias[0])

elegidas = [a["receta"] for a in sugerencias[:3]]
mostrar_lista_compras(generar_lista_compras(elegidas, inventario), elegidas)

✓ Inventario de ejemplo cargado.

📦 INVENTARIO

INGREDIENTE     CANTIDAD      VENCE        DÍAS   ESTADO
----------------------------------------------------------------------
banano          2 und         21/11/2026   -1     ❌ Vencido
tomate          3 und         23/11/2026   1      🔴 Consumir pronto
huevo           6 und         24/11/2026   2      🔴 Consumir pronto
leche           500 ml        25/11/2026   3      🔴 Consumir pronto
arepa           4 und         26/11/2026   4      🟡 Esta semana
queso           200 g         27/11/2026   5      🟡 Esta semana
papa            800 g         28/11/2026   6      🟡 Esta semana
cebolla         2 und         02/12/2026   10     🟢 En buen estado
arroz           1000 g        -            -      ⚪ No perecedero
aceite          500 ml        -            -      ⚪ No perecedero

🔴 Consume pronto estos alimentos:
   - tomate (3 und): vence mañana
   - huevo (6 und): vence en 2 días
   - leche (500 ml): vence en 3 días

❌ Estos alimentos ya se ve

## 7. Menú interactivo 🧑‍🍳
Ejecuta esta celda para usar FoodWise. Escribe el número de la opción y presiona Enter. Para salir, escribe `0`.

In [ ]:
def elegir_numeros(texto, maximo):
    """Convierte '1,3' en [1, 3], ignorando lo que no sea válido."""
    numeros = []
    for parte in texto.replace(" ", "").split(","):
        if parte.isdigit() and 1 <= int(parte) <= maximo and int(parte) not in numeros:
            numeros.append(int(parte))
    return numeros


def menu():
    ultimas_sugerencias = []
    print("🥗 Bienvenido a FoodWise: cocina con lo que ya tienes en casa")

    while True:
        print("\n" + "=" * 45)
        print(" 1. Registrar ingrediente")
        print(" 2. Ver inventario")
        print(" 3. Ver alimentos por consumir pronto")
        print(" 4. Ver recetas sugeridas")
        print(" 5. Ver detalle de una receta sugerida")
        print(" 6. Generar lista de compras")
        print(" 7. Eliminar ingrediente")
        print(" 8. Cargar inventario de ejemplo")
        print(" 0. Salir")
        opcion = input("Elige una opción: ").strip()

        if opcion == "1":
            registrar_ingrediente_usuario()
        elif opcion == "2":
            mostrar_inventario()
        elif opcion == "3":
            mostrar_por_vencer()
        elif opcion == "4":
            ultimas_sugerencias = sugerir_recetas(inventario)
            mostrar_sugerencias(ultimas_sugerencias)
        elif opcion == "5" or opcion == "6":
            if len(ultimas_sugerencias) == 0:
                ultimas_sugerencias = sugerir_recetas(inventario)
                mostrar_sugerencias(ultimas_sugerencias)
            if len(ultimas_sugerencias) == 0:
                continue
            if opcion == "5":
                numeros = elegir_numeros(input("Número de la receta: "), len(ultimas_sugerencias))
                if len(numeros) == 0:
                    print("  Número no válido.")
                else:
                    mostrar_detalle_receta(ultimas_sugerencias[numeros[0] - 1])
            else:
                numeros = elegir_numeros(input("Números de las recetas que quieres preparar (ej: 1,3): "),
                                         len(ultimas_sugerencias))
                if len(numeros) == 0:
                    print("  No elegiste ninguna receta válida.")
                else:
                    elegidas = [ultimas_sugerencias[n - 1]["receta"] for n in numeros]
                    mostrar_lista_compras(generar_lista_compras(elegidas, inventario), elegidas)
        elif opcion == "7":
            eliminar_ingrediente(input("Ingrediente a eliminar: "))
            ultimas_sugerencias = []
        elif opcion == "8":
            cargar_inventario_ejemplo()
            ultimas_sugerencias = []
        elif opcion == "0":
            print("¡Gracias por usar FoodWise! Menos desperdicio, más ahorro. 🌱")
            break
        else:
            print("  Opción no válida, intenta de nuevo.")

        if opcion == "1":
            ultimas_sugerencias = []   # el inventario cambió, se recalculan las sugerencias


inventario.clear()
menu()

🥗 Bienvenido a FoodWise: cocina con lo que ya tienes en casa

 1. Registrar ingrediente
 2. Ver inventario
 3. Ver alimentos por consumir pronto
 4. Ver recetas sugeridas
 5. Ver detalle de una receta sugerida
 6. Generar lista de compras
 7. Eliminar ingrediente
 8. Cargar inventario de ejemplo
 0. Salir
Elige una opción: Tomate
  Opción no válida, intenta de nuevo.

 1. Registrar ingrediente
 2. Ver inventario
 3. Ver alimentos por consumir pronto
 4. Ver recetas sugeridas
 5. Ver detalle de una receta sugerida
 6. Generar lista de compras
 7. Eliminar ingrediente
 8. Cargar inventario de ejemplo
 0. Salir
Elige una opción: tomate
  Opción no válida, intenta de nuevo.

 1. Registrar ingrediente
 2. Ver inventario
 3. Ver alimentos por consumir pronto
 4. Ver recetas sugeridas
 5. Ver detalle de una receta sugerida
 6. Generar lista de compras
 7. Eliminar ingrediente
 8. Cargar inventario de ejemplo
 0. Salir
Elige una opción: 1
Nombre del ingrediente: tomate
Cantidad (en und): 3
Fec